# Unit Test Generator

## Automatically Generate Unit Tests for Your Code!

This tool uses to generate comprehensive unit tests for your functions and methods, and run Python tests!

**Features:**
- Multi-language support (Python, JavaScript, TypeScript)
- Run Python tests directly in the interface (Python)

In [ ]:
import os
import sys
import subprocess
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
AVAILABLE_MODELS = {
    "🆓 Llama 3.2 (Free)": "llama3.2"
}

# Create clients dict to handle multiple providers
clients = {}

# Always add Ollama client for free model
clients["ollama"] = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

# Check if OpenAI key is valid and add OpenAI models
if api_key:
    clients["openai"] = OpenAI(api_key=api_key)
    
    # Add OpenAI models to the available models
    AVAILABLE_MODELS.update({
        "💎 GPT-4o Mini (OpenAI)": "gpt-4o-mini",
        "💎 GPT-4o (OpenAI)": "gpt-4o",
        "💎 GPT-4 Turbo (OpenAI)": "gpt-4-turbo-preview"
    })

In [ ]:
#  Supported languages

SUPPORTED_LANGUAGES = {
    "Python": {
        "framework": "unittest",
        "extension": ".py",
        "test_prefix": "test_",
        "imports": "import unittest",
        "runnable": True
    },
    "JavaScript": {
        "framework": "Jest",
        "extension": ".js",
        "test_prefix": "",
        "imports": "// Jest tests",
        "runnable": False
    },
    "TypeScript": {
        "framework": "Jest",
        "extension": ".ts",
        "test_prefix": "",
        "imports": "// Jest tests",
        "runnable": False
    }
}

print(f"Loaded {len(SUPPORTED_LANGUAGES)} programming languages")

In [ ]:
#  generate tests

def generate_unit_tests(
    code: str,
    language: str,
    model_choice: str
) -> str:
    """
    Generate unit tests for the given code.
    
    Args:
        code: Source code to generate tests for
        language: Programming language
        model_choice: model to use
    
    Returns:
        Generated test code
    """
    
    if not code or not code.strip():
        return "# Error: No code provided. Please enter code in the input section."
    
    lang_config = SUPPORTED_LANGUAGES[language]
    
    # Build prompt
    system_prompt = f"""
You are an expert software testing engineer specializing in writing comprehensive unit tests.

Generate complete, runnable unit tests for {language} code using the {lang_config['framework']} framework.

IMPORTANT:
- Generate ONLY the test code, no explanations
- Include necessary imports
- Test normal cases, edge cases, and error conditions
- Use descriptive test names
- Include assertions for expected behavior
- Follow {language} best practices
- Make tests complete and runnable
"""
    
    user_prompt = f"""
Generate comprehensive unit tests for the following {language} code:

```{language.lower()}
{code}
```

Requirements:
1. Use {lang_config['framework']} testing framework
2. Include all necessary imports
3. Test at least 3-5 different scenarios:
   - Normal/happy path cases
   - Edge cases (empty inputs, large numbers, boundaries)
   - Error cases (invalid inputs, exceptions)
4. Use clear, descriptive test method names
5. Include helpful assertions and messages

Return ONLY the test code, no markdown formatting or explanations.
"""
    
    # Get model ID
    model_id = AVAILABLE_MODELS[model_choice]
    
    # Determine which client to use based on model
    if "gpt-" in model_id or model_id.startswith("gpt"):
        # OpenAI model
        client = clients.get("openai")
        if not client:
            return """# Error: OpenAI model selected but no API key configured.
# Please add your OPENAI_API_KEY in the configuration cell, or select the free Ollama model."""
    else:
        # Ollama model (free)
        client = clients.get("ollama")
    
    try:
        # Generate tests
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_tokens=2000
        )
        
        generated_tests = response.choices[0].message.content.strip()
        
        # Clean up markdown code blocks if present
        if '```' in generated_tests:
            lines = generated_tests.split('\n')
            cleaned_lines = []
            in_code_block = False
            
            for line in lines:
                if line.strip().startswith('```'):
                    in_code_block = not in_code_block
                    continue
                if in_code_block or not line.strip().startswith('```'):
                    cleaned_lines.append(line)
            
            generated_tests = '\n'.join(cleaned_lines)
        
        return generated_tests
        
    except Exception as e:
        return f"# Error generating tests: {str(e)}"

print("Test generation engine ready")

In [ ]:
def run_python_tests(original_code: str, test_code: str) -> str:
    """
    Execute Python unit tests and return the results.
    
    Args:
        original_code: The original function/class code
        test_code: The generated test code
    
    Returns:
        Test execution results
    """
    
    if not test_code or not test_code.strip():
        return "❌ Error: No test code to run. Generate tests first!"
    
    if "Error:" in test_code[:100]:
        return "❌ Error: Cannot run tests because test generation failed. Please fix the errors first."
    
    # Create temporary file with both original code and tests
    temp_file = None
    try:
        # Create a temporary file using mkstemp (more reliable)
        fd, temp_file = tempfile.mkstemp(suffix='.py', text=True)
        
        # Write the code to the file descriptor
        with os.fdopen(fd, 'w') as f:
            # Write original code
            f.write("# Original Code\n")
            f.write(original_code)
            f.write("\n\n")
            
            # Write test code
            f.write("# Test Code\n")
            f.write(test_code)
            f.write("\n\n")
            
            # Add test runner
            f.write("\nif __name__ == '__main__':\n")
            f.write("    unittest.main(argv=[''], verbosity=2, exit=False)\n")
        
        # File is now closed and ready to execute
        result = subprocess.run(
            [sys.executable, temp_file],
            capture_output=True,
            text=True,
            timeout=10
        )
        
        # Format output
        output = []
        output.append("🧪 TEST EXECUTION RESULTS")
        output.append("=" * 60)
        output.append("")
        
        if result.returncode == 0:
            output.append("✅ ALL TESTS PASSED!")
        else:
            output.append("❌ SOME TESTS FAILED")
        
        output.append("")
        output.append("STDOUT:")
        output.append("-" * 60)
        output.append(result.stdout if result.stdout else "(no output)")
        
        if result.stderr:
            output.append("")
            output.append("STDERR:")
            output.append("-" * 60)
            output.append(result.stderr)
        
        output.append("")
        output.append("=" * 60)
        output.append(f"Exit Code: {result.returncode}")
        
        return "\n".join(output)
        
    except subprocess.TimeoutExpired:
        return "TEST EXECUTION TIMEOUT"
    except Exception as e:
        return f"TEST EXECUTION ERROR: {str(e)}"
    finally:
        # Clean up temporary file
        if temp_file and os.path.exists(temp_file):
            try:
                os.unlink(temp_file)
            except Exception:
                pass  # Ignore cleanup errors

print("✅ Test execution engine ready")

In [ ]:
#  Example

EXAMPLES = {
    "Python": {
        "Simple Math": """def add(a, b):
    \"\"\"Add two numbers.\"\"\" 
    return a + b
    """
    }
}

print(f"✅ Loaded examples for {len(EXAMPLES)} languages")

In [ ]:
#  Gradio stuff


# Custom CSS
custom_css = """
.code-section {
    border: 2px solid #e0e0e0;
    border-radius: 8px;
    padding: 10px;
}
.generate-btn {
    background: linear-gradient(90deg, #4CAF50, #45a049) !important;
    font-size: 16px !important;
}
.run-btn {
    background: linear-gradient(90deg, #2196F3, #1976D2) !important;
    font-size: 16px !important;
}
"""

# Build Gradio interface
with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="Unit Test Generator") as demo:
    
    gr.Markdown("""
    # Unit Test Generator
    
    Generate comprehensive unit tests for your code automatically!
    
    **How to use:**
    1. Select your programming language
    2. Paste or write your code
    3. Choose an AI model
    4. Click "Generate Tests"
    5. (Python only) Click "Run Tests" to execute them!
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            language_dropdown = gr.Dropdown(
                choices=list(SUPPORTED_LANGUAGES.keys()),
                value="Python",
                label="🌐 Programming Language",
                info="Select the language of your code"
            )
            
            model_dropdown = gr.Dropdown(
                choices=list(AVAILABLE_MODELS.keys()),
                value=list(AVAILABLE_MODELS.keys())[0],
                label="Model",
            )
            
            # Example selector
            example_dropdown = gr.Dropdown(
                choices=[],
                label="📝 Load Example",
                info="Optional: Load a pre-made example",
                interactive=True
            )
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📥 Your Code (Input)")
            input_code = gr.Code(
                label="Write or paste your code here",
                language="python",
                lines=20,
                value="# Write your function or class here\n# Example:\ndef add(a, b):\n    return a + b"
            )
        
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Generated Tests (Output)")
            output_tests = gr.Code(
                label="Generated unit tests will appear here",
                language="python",
                lines=20,
                value="# Click 'Generate Tests' to create unit tests for your code"
            )
    
    with gr.Row():
        generate_btn = gr.Button(
            "🧪 Generate Tests",
            variant="primary",
            size="lg",
            elem_classes="generate-btn"
        )
        run_btn = gr.Button(
            "▶️ Run Tests (Python Only)",
            variant="secondary",
            size="lg",
            elem_classes="run-btn"
        )
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📊 Test Execution Results")
            test_results = gr.Textbox(
                label="Test Output",
                lines=15,
                placeholder="Test results will appear here after clicking 'Run Tests'...",
                max_lines=30
            )
    
    
    # Update example dropdown when language changes
    def update_examples(language):
        if language in EXAMPLES:
            return gr.Dropdown(
                choices=list(EXAMPLES[language].keys()),
                value=None,
                label="📝 Load Example",
                interactive=True
            )
        return gr.Dropdown(choices=[], value=None, label="📝 No examples available", interactive=False)
    
    def load_example(language, example_name):
        if language in EXAMPLES and example_name in EXAMPLES[language]:
            return EXAMPLES[language][example_name]
        return ""
    
    def update_code_language(language):
        # Update syntax highlighting based on language
        lang_map = {
            "Python": "python",
            "JavaScript": "javascript",
            "TypeScript": "typescript"
        }
        return gr.Code(language=lang_map.get(language, "python"))
    
    # Event handlers
    language_dropdown.change(
        fn=update_examples,
        inputs=[language_dropdown],
        outputs=[example_dropdown]
    )
    
    example_dropdown.change(
        fn=load_example,
        inputs=[language_dropdown, example_dropdown],
        outputs=[input_code]
    )
    
    generate_btn.click(
        fn=generate_unit_tests,
        inputs=[input_code, language_dropdown, model_dropdown],
        outputs=[output_tests]
    )
    
    run_btn.click(
        fn=run_python_tests,
        inputs=[input_code, output_tests],
        outputs=[test_results]
    )

# Launch the app
if __name__ == "__main__":
    demo.launch(
        share=False,
        inbrowser=True,
        show_error=True
    )